# Chapter 11 - 클래스 종합 실습
파이썬 클래스
상품, 장바구니, 할인 정책, 고객을 함께 설계합니다.


1. 상품: 속성 검증과 문자열 표현


In [ ]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    @property
    def price(self):
        return self.__price

    @price.setter
    def price(self, value):
        if not isinstance(value, int) or isinstance(value, bool):
            raise TypeError('가격은 정수여야 합니다.')
        if value < 0:
            raise ValueError('가격은 0 이상이어야 합니다.')
        self.__price = value

    def __str__(self):
        return f'{self.name}: {self.price:,}원'

    def __repr__(self):
        return f'Product({self.name!r}, {self.price!r})'


2. 장바구니: 상품 객체를 여러 개 가집니다.


In [ ]:
class Cart:
    def __init__(self, products=None):
        # 전달받은 리스트를 복사해 외부 변경의 영향을 줄입니다.
        self.products = [] if products is None else list(products)

    def add(self, product):
        if not isinstance(product, Product):
            raise TypeError('Product 객체만 추가할 수 있습니다.')
        self.products.append(product)

    def total_price(self):
        return sum(product.price for product in self.products)

    def __len__(self):
        return len(self.products)

    def __bool__(self):
        return len(self) > 0

    def __str__(self):
        return f'상품 {len(self)}개 / 총액 {self.total_price():,}원'


3. 할인 정책: 같은 calculate()가 정책마다 다르게 동작합니다.


In [ ]:
class DiscountPolicy:
    def calculate(self, price):
        raise NotImplementedError('자식 클래스에서 calculate()를 구현하세요.')


class NoDiscount(DiscountPolicy):
    def calculate(self, price):
        return price

    def __str__(self):
        return '할인 없음'


class RateDiscount(DiscountPolicy):
    def __init__(self, rate):
        self.rate = rate

    @property
    def rate(self):
        return self.__rate

    @rate.setter
    def rate(self, value):
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            raise TypeError('할인율은 숫자여야 합니다.')
        if not 0 <= value <= 1:
            raise ValueError('할인율은 0 이상 1 이하여야 합니다.')
        self.__rate = value

    def calculate(self, price):
        return round(price * (1 - self.rate))

    def __str__(self):
        return f'{self.rate:.0%} 할인'


class AmountDiscount(DiscountPolicy):
    def __init__(self, amount):
        if not isinstance(amount, int) or isinstance(amount, bool):
            raise TypeError('할인 금액은 정수여야 합니다.')
        if amount < 0:
            raise ValueError('할인 금액은 0 이상이어야 합니다.')
        self.amount = amount

    def calculate(self, price):
        return max(0, price - self.amount)

    def __str__(self):
        return f'{self.amount:,}원 할인'


4. 고객: 어떤 할인 정책이 와도 같은 방식으로 사용합니다.


In [ ]:
class Customer:
    def __init__(self, name, cart=None):
        self.name = name
        self.cart = Cart() if cart is None else cart

    def add_to_cart(self, product):
        self.cart.add(product)

    def checkout(self, discount_policy):
        if not isinstance(discount_policy, DiscountPolicy):
            raise TypeError('DiscountPolicy 객체가 필요합니다.')
        if not self.cart:
            return f'{self.name}: 장바구니가 비어 있습니다.'

        final_price = discount_policy.calculate(self.cart.total_price())
        return f'{self.name} / {discount_policy} / 결제 금액 {final_price:,}원'


5. 같은 장바구니로 전체 흐름을 확인합니다.


In [ ]:
keyboard = Product('키보드', 35_000)
mouse = Product('마우스', 18_000)

demo_cart = Cart([keyboard, mouse])
customer = Customer('민수', demo_cart)

print(keyboard)
print(repr(mouse))
print(customer.cart)
print()

policies = [
    NoDiscount(),
    RateDiscount(0.1),
    AmountDiscount(5_000),
]

for policy in policies:
    print(customer.checkout(policy))
print()


6. 객체가 서로 독립적인지 확인합니다.


In [ ]:
another_customer = Customer('지수')
print(customer.cart is another_customer.cart)
print(another_customer.checkout(NoDiscount()))
print()


7. 잘못된 값은 객체가 스스로 막습니다.


In [ ]:
try:
    Product('잘못된 상품', -1)
except ValueError as error:
    print(type(error).__name__, error)

try:
    RateDiscount(1.5)
except ValueError as error:
    print(type(error).__name__, error)

try:
    customer.checkout('10% 할인')
except TypeError as error:
    print(type(error).__name__, error)


### 정리와 전이

8. 핵심 정리
- Product는 자기 가격을 검증합니다.
- Cart는 Product 객체를 가집니다.                    -> 컴포지션
- 할인 정책들은 DiscountPolicy를 상속합니다.         -> 상속
- 같은 calculate()가 정책마다 다르게 동작합니다.     -> 오버라이딩과 다형성
- 새 할인 정책은 Customer를 고치지 않고 새 클래스로 추가할 수 있습니다.
